# 02o3: Extracting Data with LangChain (Alternative Approach)

This notebook demonstrates an alternative approach to extracting entities and relationships from NotePlan notes using LangChain's structured output capabilities instead of agents.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup

All environment detection, Neo4j connection, and NotePlan directory configuration are handled in `00-import.ipynb`.

## Overview

This notebook provides an alternative to the agent-based extraction notebooks using LangChain's structured output features. Both approaches achieve the same goal - extracting entities and relationships from your notes - but use different methods. Choose this approach if you prefer LangChain's Pydantic-based structured outputs over agent-based extraction.

**Agent-Based Approaches (Recommended):**
- [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb): Extract data from a **single file** using agents
- [**02o2-extracting-data.ipynb**](./02o2-extracting-data.ipynb): Extract data from **multiple files** in bulk using agents

**This Notebook (LangChain Alternative):**
- Uses LangChain's Pydantic output parsers for structured extraction
- Creates prompt templates for entity/relationship extraction
- Uses ChatOpenAI (via LiteLLM proxy) with structured outputs
- Extracts entities and relationships using LangChain chains

We'll:
1. Use LangChain's Pydantic output parsers for structured extraction
2. Create prompt templates for entity/relationship extraction
3. Use ChatOpenAI (via LiteLLM proxy) with structured outputs
4. Extract entities and relationships using LangChain chains
5. Compare with agent-based approach from notebooks 02o1 and 02o2


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
import asyncio
from typing import List

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnableLambda

# Import our graph types
from knowledge_agents.types.graph import Entity, Relationship, GraphBuilderAgentOutput

# NotePlan utilities
from notes.traversal import get_files_from_last_month
from notes.parser import read_noteplan_file
from notes.filter import should_skip_file

print("✅ Additional libraries imported")


## Set Up LangChain LLM

Configure ChatOpenAI to use LiteLLM proxy.


In [ ]:
# Create ChatOpenAI instance using LiteLLM proxy
proxy_base_url = f"http://{settings.litellm_proxy_host}:{settings.litellm_proxy_port}"

llm = ChatOpenAI(
    model=settings.litellm_proxy_model or "lm_studio/qwen3-coder-30b",
    base_url=f"{proxy_base_url}/v1",
    api_key=settings.openai_api_key or "not-needed",
    temperature=0,
)

print("✅ LangChain LLM configured")
print(f"   Proxy URL: {proxy_base_url}")
print(f"   Model: {llm.model_name}")


## Create Extraction Prompt Template

Define a prompt template for extracting entities and relationships.


In [ ]:
# Create Pydantic output parser
parser = PydanticOutputParser(pydantic_object=GraphBuilderAgentOutput)

# Create prompt template
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at extracting structured information from text.
Extract entities (people, places, concepts, projects, etc.) and relationships between them.

Guidelines:
- Extract meaningful entities that represent important concepts, people, projects, or topics
- Avoid extracting duplicate entities
- Be selective - focus on the most important entities and relationships
- For relationships, identify how entities are connected (e.g., WORKS_ON, MENTIONS, RELATED_TO, PART_OF)
- Include any relevant properties for entities (e.g., dates, status, type)

{format_instructions}"""),
    ("human", """Extract entities and relationships from the following text:

File: {file_path}

Text:
{note_content}

Return a JSON object with entities, relationships, and insights.""")
])

# Format the prompt with parser instructions
prompt = extraction_prompt.partial(format_instructions=parser.get_format_instructions())

print("✅ Extraction prompt template created")


## Create Extraction Chain

Build a LangChain chain that combines prompt, LLM, and parser.


In [ ]:
# Create extraction chain
extraction_chain = prompt | llm | StrOutputParser() | parser

print("✅ Extraction chain created")


## Extract from NotePlan Files

Load NotePlan files and extract entities/relationships using the LangChain chain.


In [ ]:
# Get NotePlan files
files = get_files_from_last_month(NOTEPLAN_DIR)
files = [(fp, mod_time) for fp, mod_time in files if not should_skip_file(fp)]
files = files[:5]  # Limit for demo

print(f"Processing {len(files)} files")

# Extract entities and relationships using LangChain
extracted_data = []

for file_path, mod_time in files:
    relative_path = str(file_path.relative_to(NOTEPLAN_DIR))
    print(f"Processing: {relative_path}")
    
    try:
        content = read_noteplan_file(file_path)
        
        # Use LangChain chain to extract
        result = extraction_chain.invoke({
            "file_path": relative_path,
            "note_content": content[:2000]  # Limit content length
        })
        
        extracted_data.append((relative_path, result))
        print(f"  ✅ Extracted {len(result.entities)} entities, {len(result.relationships)} relationships")
    except Exception as e:
        print(f"  ❌ Error: {e}")

print(f"\n✅ Processed {len(extracted_data)} files successfully")


## Display Extracted Data

View the extracted entities and relationships.


In [ ]:
# Collect all entities and relationships
all_entities = []
all_relationships = []

for file_path, output in extracted_data:
    for entity in output.entities:
        all_entities.append({
            "file": file_path,
            "name": entity.name,
            "type": entity.type,
            "properties": entity.properties
        })
    for rel in output.relationships:
        all_relationships.append({
            "file": file_path,
            "from": rel.from_entity,
            "type": rel.type,
            "to": rel.to_entity,
            "properties": rel.properties
        })

# Display entities
if all_entities:
    print("Extracted Entities:")
    df_entities = pd.DataFrame(all_entities)
    print(df_entities.head(20))
    print(f"\nTotal entities: {len(all_entities)}")

# Display relationships
if all_relationships:
    print("\nExtracted Relationships:")
    df_relationships = pd.DataFrame(all_relationships)
    print(df_relationships.head(20))
    print(f"\nTotal relationships: {len(all_relationships)}")


## Comparison with Agent-Based Approach

**LangChain Approach (this notebook - 02o3):**
- Uses LangChain chains and structured output parsers
- More explicit control over the extraction pipeline
- Easier to customize prompts and parsing logic
- Good for simpler extraction tasks
- Direct LLM calls without agent framework overhead

**Agent-Based Approaches:**
- [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb): Single file extraction using agents
- [**02o2-extracting-data.ipynb**](./02o2-extracting-data.ipynb): Bulk file extraction using agents
- Uses the repository's graph builder agent
- More flexible and can handle complex reasoning
- Better for tasks requiring multi-step reasoning
- Integrated with repository's agent framework
- Includes caching, logging, and data persistence utilities

**When to Use Which:**
- **Use LangChain (02o3)**: If you prefer explicit prompt control and simpler pipeline
- **Use Agents (02o1/02o2)**: If you need complex reasoning, want integrated caching/logging, or prefer the agent framework

Both approaches produce the same output format (`GraphBuilderAgentOutput`), so you can use either depending on your needs.
